# 06 — Calibration-only anomaly models

This notebook fits every detector without fault or ticket labels. The main
candidates are deliberately compact:

- empirical one-metric tail evidence (`rapid_residual`);
- corroborated evidence from two distinct metrics, using both the weaker
  metric and the mean of the two strongest metric tails;
- directional CUSUM for sustained drift (`drift_cusum`);
- a metric-balanced, direction-aware temporal Isolation Forest;
- the same Isolation Forest confirmed by interpretable tail evidence; and
- an entity-calibrated Isolation Forest score with a pooled fallback.

Residuals use a frozen robust reference,
`z = (x - median) / (IQR / 1.349)`, with declared scale floors. Before a
residual enters the primary detector, it is oriented so that larger always
means more adverse: `z`, `-z`, or `abs(z)` for high-bad, low-bad, or
two-sided metrics. Isolation Forest receives at most a fixed number of
features per metric and a sample balanced across entity-days. This prevents
one densely engineered metric or one long stable regime from dominating.

Topology evidence is still calculated for localisation and diagnostic
corroboration. Peer evidence is one-sided after adverse orientation and is
restricted to comparable physical measurements. Group evidence requires a
breadth beyond its calibration null for the same topology-size band. It is
shared-scope anomaly evidence, not a claim that a splitter has failed.

Early calibration fits references and models. The later calibration slice
is split again: its first half estimates block-max thresholds and its second
half independently checks label-free incident workload. Development is only
scored here; its labels remain unopened until Notebook 07.

## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import joblib
import shutil
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.detectors import (
    calibration_thresholds,
    eligible_peer_levels,
    fit_contextual_isolation_forest,
    fit_residual_bundle,
    fit_topology_reference,
    merge_score_files,
    physical_hierarchy,
    score_residual_file,
    score_partition_file,
    split_feature_file_by_time,
    score_topology_file,
)
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    load_config,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Model fitting is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_features = os.getenv("PON_FEATURE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_eda = os.getenv("PON_EDA_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET]
)
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", legacy_features or f"{DATASET}_features_v4"
)
EDA_RUN_ID = os.getenv(
    "TELCO_EDA_RUN_ID", legacy_eda or f"{DATASET}_calibration_eda_v5"
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", f"{DATASET}_models_v8"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
FEATURE_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID
EDA_ROOT = DATA_ROOT / "eda" / DATASET / EDA_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "models" / DATASET / MODEL_RUN_ID

ALERT_POLICY = load_config("alert_policy", project_root=PROJECT_ROOT)
FEATURE_CONFIG = load_config("features", project_root=PROJECT_ROOT)
TOPOLOGY_POLICY = load_config("topology", project_root=PROJECT_ROOT)
MODEL_CONFIG = load_config("model", project_root=PROJECT_ROOT)
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
core_manifest = read_json(CORE_ROOT / "manifest.json")
feature_manifest = read_json(FEATURE_ROOT / "feature_manifest.json")
eda_decisions = read_json(EDA_ROOT / "eda_decisions.json")
CURRENT_FEATURES_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "features.py"
)
CURRENT_DETECTORS_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py"
)
CURRENT_FEATURE_CONFIG_SHA256 = file_sha256(
    PROJECT_ROOT / "configs" / "features.yml"
)
MODEL_CONFIG_SHA256 = file_sha256(PROJECT_ROOT / "configs" / "model.yml")
ALERT_POLICY_SHA256 = file_sha256(
    PROJECT_ROOT / "configs" / "alert_policy.yml"
)
require_same(
    feature_manifest,
    core_fingerprint=core_manifest["fingerprint"],
    features_module_sha256=CURRENT_FEATURES_SHA256,
    history_windows_seconds=FEATURE_CONFIG["temporal"]["history_windows_seconds"],
    lag_windows_seconds=FEATURE_CONFIG["temporal"]["lag_windows_seconds"],
    activity_windows_seconds=FEATURE_CONFIG["temporal"]["activity_windows_seconds"],
    history_metric_ids=FEATURE_CONFIG["temporal"]["history_metric_ids"],
    lag_metric_ids=FEATURE_CONFIG["temporal"]["lag_metric_ids"],
    activity_metric_ids=FEATURE_CONFIG["temporal"]["activity_metric_ids"],
    minimum_window_fraction=FEATURE_CONFIG["temporal"]["minimum_window_fraction"],
)
if feature_manifest.get("feature_config_sha256") != CURRENT_FEATURE_CONFIG_SHA256:
    print("Reusing features: the removed model block did not affect feature values")
SCRATCH_PARENT = Path(os.getenv(
    "TELCO_WORK_ROOT",
    "/content" if "google.colab" in sys.modules else tempfile.gettempdir(),
))
SCRATCH_PARENT.mkdir(parents=True, exist_ok=True)
SCRATCH_FREE_GB = shutil.disk_usage(SCRATCH_PARENT).free / 1024**3
MINIMUM_SCRATCH_GB = float(os.getenv("TELCO_MIN_SCRATCH_GB", "4"))
if SCRATCH_FREE_GB < MINIMUM_SCRATCH_GB:
    raise OSError(
        f"Notebook 06 needs at least {MINIMUM_SCRATCH_GB:.0f} GB free in "
        f"{SCRATCH_PARENT}; only {SCRATCH_FREE_GB:.1f} GB is available. "
        "Set TELCO_WORK_ROOT to a larger local disk."
    )

if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Model fitting must use the truth-unmounted run")

feature_paths = {
    name: FEATURE_ROOT / values["features"]
    for name, values in feature_manifest["partitions"].items()
}
display(pd.Series({
    "calibration_fit_features": str(feature_paths["calibration_fit"]),
    "calibration_threshold_features": str(feature_paths["calibration_threshold"]),
    "dataset": DATASET,
    "development_features": str(feature_paths["development"]),
    "model_output": str(OUTPUT_ROOT),
    "local_scratch": str(SCRATCH_PARENT),
    "local_scratch_free_gb": round(SCRATCH_FREE_GB, 1),
    "duckdb_memory_limit": os.getenv(
        "TELCO_MODEL_DUCKDB_MEMORY_LIMIT", "1GB"
    ),
    "duckdb_threads": int(os.getenv("TELCO_MODEL_DUCKDB_THREADS", "1")),
}, name="value").to_frame())


## 2. Fit the frozen self-history reference


In [ ]:
MAX_TRAINING_ROWS = int(os.getenv("TELCO_MAX_TRAINING_ROWS", "150000"))
SAMPLE_POLICY = MODEL_CONFIG["calibration_sample"]
ISOLATION_POLICY = MODEL_CONFIG["isolation_forest"]
REFERENCE_POLICY = MODEL_CONFIG["robust_reference"]
ISOLATION_TREES = int(os.getenv(
    "TELCO_ISOLATION_TREES", str(ISOLATION_POLICY["n_estimators"])
))
ISOLATION_MAX_SAMPLES = int(ISOLATION_POLICY["max_samples"])
ISOLATION_MAX_FEATURES = float(ISOLATION_POLICY["max_features"])
ISOLATION_FEATURES_PER_METRIC = int(
    ISOLATION_POLICY["maximum_features_per_metric"]
)
ENTITY_SCORE_POLICY = ISOLATION_POLICY["entity_score_calibration"]
ROWS_PER_ENTITY_DAY = int(SAMPLE_POLICY["maximum_rows_per_entity_day"])
MODEL_SEED = int(ISOLATION_POLICY["random_seed"])

required_catalogue_columns = {
    "metric_id", "expected_cadence_seconds", "direction", "peer_eligible",
}
missing_catalogue_columns = required_catalogue_columns - set(catalogue)
if missing_catalogue_columns:
    raise ValueError(
        f"Metric catalogue is missing {sorted(missing_catalogue_columns)}"
    )

model_cadences = pd.to_numeric(
    catalogue["expected_cadence_seconds"], errors="coerce"
).dropna().unique()
if len(model_cadences) != 1:
    raise ValueError(
        "Notebook 06 requires one prepared modelling cadence; "
        "resample mixed-cadence sources during feature engineering."
    )
CADENCE_SECONDS = float(model_cadences[0])

FEATURE_MANIFEST_SHA256 = file_sha256(FEATURE_ROOT / "feature_manifest.json")
TOPOLOGY_CONFIG_SHA256 = file_sha256(PROJECT_ROOT / "configs" / "topology.yml")
expected_model_inputs = {
    "core_fingerprint": core_manifest["fingerprint"],
    "feature_manifest_sha256": FEATURE_MANIFEST_SHA256,
    "detectors_module_sha256": CURRENT_DETECTORS_SHA256,
    "model_config_sha256": MODEL_CONFIG_SHA256,
    "alert_policy_sha256": ALERT_POLICY_SHA256,
    "topology_config_sha256": TOPOLOGY_CONFIG_SHA256,
    "eda_decisions_sha256": file_sha256(EDA_ROOT / "eda_decisions.json"),
}

if OUTPUT_ROOT.exists():
    model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
    require_same(model_manifest, **expected_model_inputs)
    if file_sha256(OUTPUT_ROOT / "resolved_policy.json") != model_manifest["resolved_policy_sha256"]:
        raise ValueError("The frozen model policy no longer matches its manifest")
    bundle = joblib.load(OUTPUT_ROOT / "residual_bundle.joblib")
    print("Using existing immutable model:", OUTPUT_ROOT)
else:
    bundle = fit_residual_bundle(
        feature_paths["calibration_fit"],
        use_entity_reference=True,
        catalogue=catalogue,
        reference_exclusions=eda_decisions.get("reference_exclusions", []),
        maximum_training_rows=MAX_TRAINING_ROWS,
        maximum_rows_per_entity_day=ROWS_PER_ENTITY_DAY,
        random_seed=MODEL_SEED,
        isolation_trees=ISOLATION_TREES,
        isolation_max_samples=ISOLATION_MAX_SAMPLES,
        isolation_max_features=ISOLATION_MAX_FEATURES,
        maximum_isolation_features_per_metric=ISOLATION_FEATURES_PER_METRIC,
        entity_reference_minimum_rows=REFERENCE_POLICY["minimum_entity_rows"],
        isolation_entity_minimum_rows=ENTITY_SCORE_POLICY["minimum_rows"],
        isolation_entity_scale_floor_fraction=(
            ENTITY_SCORE_POLICY["scale_floor_fraction_of_global"]
        ),
        fit_multivariate=True,
    )


def current_topology_features(metric_catalogue, fitted_features):
    """Select comparable current-state features explicitly approved for peers."""

    configured = set(TOPOLOGY_POLICY["peer_policy"]["metric_ids"])
    declared = set(
        metric_catalogue.loc[
            metric_catalogue["peer_eligible"].eq(True), "metric_id"
        ].astype(str)
    )
    eligible_metrics = configured & declared
    current_suffixes = (
        "__level", "__nonzero", "__positive_log10",
        "__positive_log1p", "__increment", "__reset",
        "__state", "__transition",
    )
    return sorted(
        feature for feature in fitted_features
        if feature.rsplit("__", 1)[0] in eligible_metrics
        and feature.endswith(current_suffixes)
    )


TOPOLOGY_FEATURES = current_topology_features(
    catalogue, bundle["feature_columns"]
)

display(pd.Series({
    "training_rows": bundle["training_rows"],
    "training_sample": bundle["training_sample_strategy"],
    "maximum_rows_per_entity_day": bundle["maximum_rows_per_entity_day"],
    "health_features": len(bundle["feature_columns"]),
    "entity_reference": bundle["use_entity_reference"],
    "entity_reference_entities": (
        len(bundle["entity_centre"])
        if bundle["entity_centre"] is not None else 0
    ),
    "minimum_entity_reference_rows": (
        int(bundle["entity_reference_counts"].min().min())
        if bundle.get("entity_reference_counts") is not None else 0
    ),
    "empirical_tail_calibration": bool(bundle["tail_reference"]),
    "base_isolation_features": len(bundle["isolation_base_features"]),
    "temporal_isolation_features": len(bundle["isolation_temporal_features"]),
    "isolation_forest_fitted": bundle["isolation_forest_temporal"] is not None,
    "topology_residual_features": len(TOPOLOGY_FEATURES),
}, name="value").to_frame())

In [ ]:
display(pd.DataFrame({"topology_feature": TOPOLOGY_FEATURES}))
display(bundle["isolation_feature_audit"])

## 3. Define the single partition-scoring path


In [ ]:
# Topology order comes from the canonical hierarchy, not a second manual list.
peer_policy = TOPOLOGY_POLICY["peer_policy"]
group_policy = TOPOLOGY_POLICY["group_policy"]


## 4. Fit topology evidence, score calibration/development, and freeze thresholds

Topology is fitted on early calibration and then applied unchanged. It is
retained as localisation/corroboration evidence, not automatically OR-ed
into a deployable detector. The late-calibration score file is split by
time so threshold estimation and workload verification are independent.

In [ ]:
if not OUTPUT_ROOT.exists():
    with immutable_output_directory(OUTPUT_ROOT) as output:
        topology_path = CORE_ROOT / "topology_memberships.parquet"
        topology_enabled = topology_path.exists()
        topology_reason = "available" if topology_enabled else "topology file unavailable"
        topology = pd.read_parquet(topology_path) if topology_enabled else None
        if topology_enabled and not TOPOLOGY_FEATURES:
            topology_enabled = False
            topology_reason = "no approved peer-comparable current-state features"

        topology_reference = pd.DataFrame()
        peer_level = None
        group_levels = []
        contextual_bundle = None
        contextual_status = "topology unavailable"

        # Fit all references only on early calibration.
        with tempfile.TemporaryDirectory(
            dir=SCRATCH_PARENT, prefix="telco-model-fit-"
        ) as fit_name:
            fit_workspace = Path(fit_name)
            fit_self = fit_workspace / "calibration_fit_self.parquet"
            fit_residuals = fit_workspace / "calibration_fit_residuals.parquet"
            score_residual_file(
                bundle,
                feature_paths["calibration_fit"],
                fit_self,
                cadence_seconds=CADENCE_SECONDS,
                dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
                cusum_allowance=ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
                residual_destination=fit_residuals if topology_enabled else None,
                residual_features=TOPOLOGY_FEATURES,
            )

            if topology_enabled:
                physical = topology.loc[
                    topology["group_family"].eq("physical_topology")
                ]
                try:
                    peer_candidates = eligible_peer_levels(
                        physical,
                        minimum_valid_peers=peer_policy["minimum_valid_peers"],
                        minimum_entity_coverage=peer_policy["minimum_entity_coverage"],
                    )
                    if not peer_candidates:
                        raise ValueError("No physical level has adequate peer membership")
                    group_levels = physical_hierarchy(physical)
                    topology_reference = fit_topology_reference(
                        fit_residuals,
                        topology,
                        TOPOLOGY_FEATURES,
                        peer_group_type=peer_candidates,
                        group_types=group_levels,
                        min_peers=TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                        min_group_entities=TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                        min_group_fraction=group_policy["minimum_available_fraction"],
                        affected_fraction_quantile=group_policy["affected_fraction_quantile"],
                    )
                    fitted_peer_levels = topology_reference.loc[
                        topology_reference["channel"].eq("peer_deviation"),
                        "group_type",
                    ].drop_duplicates().tolist()
                    if len(fitted_peer_levels) != 1:
                        raise ValueError("Topology fit did not select one peer level")
                    peer_level = fitted_peer_levels[0]
                    TOPOLOGY_FEATURES = sorted(
                        topology_reference["leading_feature"].astype(str).unique()
                    )
                except ValueError as error:
                    topology_enabled = False
                    topology_reason = str(error)
                    topology_reference = pd.DataFrame()
                    peer_level = None
                    group_levels = []

            fit_policy = {
                "cusum_allowance": ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
                "topology_enabled": topology_enabled,
                "peer_group_type": peer_level,
                "group_types": group_levels,
                "min_peers": TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                "min_group_entities": TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                "min_group_fraction": TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
                "affected_fraction_quantile": group_policy["affected_fraction_quantile"],
                "topology_features": TOPOLOGY_FEATURES,
            }

            if topology_enabled:
                fit_topology = fit_workspace / "calibration_fit_topology.parquet"
                fit_combined = fit_workspace / "calibration_fit_combined.parquet"
                score_topology_file(
                    fit_residuals,
                    topology,
                    topology_reference,
                    fit_topology,
                    peer_group_type=peer_level,
                    group_types=group_levels,
                    min_peers=fit_policy["min_peers"],
                    min_group_entities=fit_policy["min_group_entities"],
                    min_group_fraction=fit_policy["min_group_fraction"],
                )
                fit_residuals.unlink()
                merge_score_files(fit_self, fit_topology, fit_combined)
                fit_self.unlink()
                fit_topology.unlink()
                try:
                    contextual_bundle = fit_contextual_isolation_forest(
                        fit_combined,
                        ISOLATION_POLICY["contextual_inputs"],
                        maximum_training_rows=MAX_TRAINING_ROWS,
                        random_seed=MODEL_SEED,
                        trees=ISOLATION_TREES,
                        maximum_samples=ISOLATION_MAX_SAMPLES,
                        maximum_features=ISOLATION_MAX_FEATURES,
                    )
                    contextual_status = "available"
                except ValueError as error:
                    contextual_status = f"unavailable: {error}"

        resolved_policy = {
            "alert_policy": ALERT_POLICY,
            "topology_policy": TOPOLOGY_POLICY,
            "model_policy": MODEL_CONFIG,
            **fit_policy,
            "contextual_isolation_status": contextual_status,
        }

        # Score the late calibration slice once, split its output, then score
        # development through the identical frozen path.
        with tempfile.TemporaryDirectory(
            dir=SCRATCH_PARENT, prefix="telco-model-score-"
        ) as score_name:
            score_workspace = Path(score_name)
            late_scores = score_workspace / "calibration_late_scores.parquet"
            score_partition_file(
                bundle,
                feature_paths["calibration_threshold"],
                late_scores,
                score_workspace / "work_calibration",
                cadence_seconds=CADENCE_SECONDS,
                dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
                resolved_policy=resolved_policy,
                topology=topology,
                topology_reference=topology_reference,
                contextual_isolation_bundle=contextual_bundle,
            )

            threshold_scores = score_workspace / "calibration_threshold_scores.parquet"
            verification_scores = score_workspace / "calibration_verification_scores.parquet"
            late_split = split_feature_file_by_time(
                late_scores,
                threshold_scores,
                verification_scores,
                fit_fraction=0.50,
            )
            late_scores.unlink()

            development_scores = score_workspace / "development_scores.parquet"
            score_partition_file(
                bundle,
                feature_paths["development"],
                development_scores,
                score_workspace / "work_development",
                cadence_seconds=CADENCE_SECONDS,
                dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
                resolved_policy=resolved_policy,
                topology=topology,
                topology_reference=topology_reference,
                contextual_isolation_bundle=contextual_bundle,
            )

            score_files = {
                "calibration_threshold": threshold_scores,
                "calibration_verification": verification_scores,
                "development": development_scores,
            }
            published_scores = {}
            for name, source in score_files.items():
                destination = output / source.name
                shutil.copy2(source, destination)
                published_scores[name] = destination.name

            score_columns = set(
                pq.ParquetFile(threshold_scores).schema_arrow.names
            )
            channel_order = (
                "rapid_residual", "multimetric_residual",
                "multimetric_tail_mean", "drift_cusum",
                "isolation_forest_temporal", "isolation_forest_confirmed",
                "isolation_forest_entity_calibrated",
                "isolation_forest_base", "peer_deviation", "group_common_mode",
                "dispersion_change", "pca_spe", "isolation_forest_contextual",
            )
            channels = [name for name in channel_order if name in score_columns]
            print("Calibrating thresholds from the first late-calibration half")
            thresholds = calibration_thresholds(
                threshold_scores,
                ALERT_POLICY["thresholds"]["candidate_quantiles"],
                block_column="entity_id",
                block_duration_seconds=ALERT_POLICY["thresholds"]["block_seconds"],
                minimum_block_rows=max(4, round(0.5 * 86_400 / CADENCE_SECONDS)),
                model_ids=channels,
            )

        bundle["contextual_isolation_bundle"] = contextual_bundle
        joblib.dump(bundle, output / "residual_bundle.joblib")
        bundle["feature_audit"].to_parquet(
            output / "feature_retention.parquet", index=False
        )
        bundle["isolation_feature_audit"].to_parquet(
            output / "isolation_feature_retention.parquet", index=False
        )
        if contextual_bundle is not None:
            contextual_bundle["feature_audit"].to_parquet(
                output / "contextual_feature_retention.parquet", index=False
            )
        thresholds.to_parquet(
            output / "calibration_thresholds.parquet", index=False
        )
        if len(topology_reference):
            topology_reference.to_parquet(
                output / "topology_reference.parquet", index=False
            )
        write_json(output / "resolved_policy.json", resolved_policy)

        model_manifest = {
            "dataset": DATASET,
            **expected_model_inputs,
            "resolved_policy_sha256": file_sha256(output / "resolved_policy.json"),
            "calibration_only_fit": True,
            "calibration_fit_fraction": feature_manifest["calibration_fit_fraction"],
            "calibration_threshold_start": feature_manifest["calibration_threshold_start"],
            "calibration_verification_start": late_split["cutoff"],
            "truth_files_read": [],
            "channels": channels,
            "primary_channels": [
                "rapid_residual", "multimetric_residual",
                "multimetric_tail_mean", "drift_cusum",
                "isolation_forest_temporal", "isolation_forest_confirmed",
                "isolation_forest_entity_calibrated",
            ],
            "localisation_channels": ["peer_deviation", "group_common_mode"],
            "challenger_channels": [
                "isolation_forest_base", "dispersion_change", "pca_spe",
                "isolation_forest_contextual",
            ],
            "score_files": published_scores,
            "feature_columns": bundle["feature_columns"],
            "base_isolation_features": bundle["isolation_base_features"],
            "temporal_isolation_features": bundle["isolation_temporal_features"],
            "contextual_isolation_features": (
                contextual_bundle["feature_columns"]
                if contextual_bundle is not None else []
            ),
            "topology_enabled": topology_enabled,
            "topology_status": topology_reason,
            "topology_residual_features": TOPOLOGY_FEATURES,
            "peer_level": peer_level,
            "cadence_seconds": CADENCE_SECONDS,
            "dispersion_window_seconds": feature_manifest["dispersion_window_seconds"],
            "seasonal_periods": feature_manifest["seasonal_periods"],
            "threshold_method": (
                "daily block-max quantiles from the first half of late calibration"
            ),
            "workload_verification": (
                "independent second half of late calibration"
            ),
        }
        write_json(output / "model_manifest.json", model_manifest)

thresholds = pd.read_parquet(OUTPUT_ROOT / "calibration_thresholds.parquet")
display(thresholds)

## 5. Acceptance


In [ ]:
model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
assert model_manifest["calibration_only_fit"] is True
assert model_manifest["truth_files_read"] == []
assert {
    "rapid_residual", "multimetric_residual", "multimetric_tail_mean",
    "isolation_forest_temporal", "isolation_forest_confirmed",
    "isolation_forest_entity_calibrated",
} <= set(model_manifest["channels"])
assert {
    "calibration_threshold", "calibration_verification", "development",
} == set(model_manifest["score_files"])

display(pd.Series({
    "primary_channels_available": [
        name for name in model_manifest["primary_channels"]
        if name in model_manifest["channels"]
    ],
    "topology_status": model_manifest["topology_status"],
    "localisation_channels_available": [
        name for name in model_manifest["localisation_channels"]
        if name in model_manifest["channels"]
    ],
    "challengers_available": [
        name for name in model_manifest["challenger_channels"]
        if name in model_manifest["channels"]
    ],
}, name="result").to_frame())
print("PASS — frozen scores, independent thresholds and workload verification are ready")
print("Next: 07_CHALLENGER_MODELS.ipynb")